# GatedWave vs minGRU — Character-level LM on Colab T4

**What this tests**: Does GatedWave (multi-scale oscillatory RNN) outperform minGRU on character-level language modelling?

**Configs**:
- **A**: GatedWave, default theta (0.01→1.0)
- **B**: GatedWave, language-tuned theta (0.063→1.257, covers word→sentence scale)
- **C**: minGRU baseline (parameter-matched)
- **D**: GatedWave, language-tuned theta + log-spacing (H11 follow-up)

**Scale for T4**: ~10M params each, 20k steps. Primary comparison A vs C. Full-scale (80M, 50k steps) requires A100.

**Estimated time on T4**: ~1.5–2.5h per config, ~6–10h total for A+B+C+D.

In [ ]:
# ── 1. Runtime check ─────────────────────────────────────────────────────────
import subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('GPU:', result.stdout.strip())
else:
    print('No GPU detected — switch to T4 runtime: Runtime > Change runtime type > T4 GPU')
    sys.exit(1)

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────────
# JAX CUDA build + Flax + Optax. Takes ~2 min.
!pip install -q -U "jax[cuda12]" flax optax

import jax
print('JAX version:', jax.__version__)
print('Devices:', jax.devices())

In [ ]:
# ── 3. Get the code ───────────────────────────────────────────────────────────
#
# Option A (recommended): mount Google Drive, copy files from there
#   1. Save train_char_lm.py and gated_wave.py to your Drive
#   2. Run the Drive mount cell below
#
# Option B: upload directly via Files panel (left sidebar → upload icon)
#   Then skip the Drive mount cell.
#
# Option C: clone from GitHub if the repo is public
#   !git clone https://github.com/hebenon/continuous-language.git .

# ----- Option A: Mount Drive -----
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

# Edit this path to wherever you saved the files on Drive:
DRIVE_CODE_PATH = '/content/drive/MyDrive/continuous-language'

for fname in ['train_char_lm.py', 'gated_wave.py']:
    src = os.path.join(DRIVE_CODE_PATH, fname)
    shutil.copy(src, f'/content/{fname}')
    print(f'Copied {fname}')

In [ ]:
# ── 4. Download TinyShakespeare ───────────────────────────────────────────────
import os
os.makedirs('data/shakespeare', exist_ok=True)

!wget -q -O data/shakespeare/input.txt \
    https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

size = os.path.getsize('data/shakespeare/input.txt')
print(f'TinyShakespeare: {size:,} bytes ({size/1e6:.1f}MB)')

In [ ]:
# ── 5. Stability check (5k steps, ~20–30 min) ─────────────────────────────────
# Run this first. If loss explodes or stays flat → architectural problem.
# Expected: loss should decline from ~4.2 toward ~2.0 within 5k steps.

!python train_char_lm.py \
    --model gated_wave \
    --d_model 512 --n_layers 6 --n_scales 4 \
    --steps 5000 --eval_every 500 --save_every 1000 \
    --data data/shakespeare/input.txt \
    --save_dir checkpoints/stability

---
## Main Experiments

If stability check looks good (smooth loss decline), run A and C first — that's the core question.
B and D refine the theta initialisation hypothesis; run them after A+C if time permits.

**Interpretation guide**:
| Pattern | Meaning |
|---------|--------|
| A ≈ B > C | Architecture wins; theta init doesn't matter |
| B > A > C | Architecture + theta init both matter |
| C ≥ A ≈ B | GatedWave gains nothing — rethink architecture |
| A diverges | Gradient instability — lower `--lr` |
| A > C but B < A | Theta range 0.063–1.257 wrong for char-LM |

In [ ]:
# ── 6. Config A — GatedWave default theta (0.01→1.0) ─────────────────────────
# ~10M params | ~1.5–2.5h on T4

!python train_char_lm.py \
    --model gated_wave \
    --d_model 512 --n_layers 6 --n_scales 4 \
    --steps 20000 --eval_every 500 --save_every 2000 \
    --data data/shakespeare/input.txt \
    --save_dir checkpoints/config_A

In [ ]:
# ── 7. Config C — minGRU baseline (parameter-matched ~10.7M) ─────────────────
# ~1.5–2.5h on T4

!python train_char_lm.py \
    --model mingru \
    --d_model 900 --n_layers 6 \
    --steps 20000 --eval_every 500 --save_every 2000 \
    --data data/shakespeare/input.txt \
    --save_dir checkpoints/config_C

In [ ]:
# ── 8. Config B — GatedWave language-tuned theta, linear spacing ──────────────
# theta_min=0.063 (period~100 chars = sentence scale)
# theta_max=1.257 (period~5 chars = word scale)
# ~1.5–2.5h on T4

!python train_char_lm.py \
    --model gated_wave \
    --d_model 512 --n_layers 6 --n_scales 4 \
    --theta_min 0.063 --theta_max 1.257 \
    --steps 20000 --eval_every 500 --save_every 2000 \
    --data data/shakespeare/input.txt \
    --save_dir checkpoints/config_B

In [ ]:
# ── 9. Config D — GatedWave language-tuned theta, log spacing (H11) ──────────
# Log-spaced: 0.063 / 0.19 / 0.56 / 1.26 — covers sentence/clause/phrase/word
# ~1.5–2.5h on T4

!python train_char_lm.py \
    --model gated_wave \
    --d_model 512 --n_layers 6 --n_scales 4 \
    --theta_min 0.063 --theta_max 1.257 --log_theta \
    --steps 20000 --eval_every 500 --save_every 2000 \
    --data data/shakespeare/input.txt \
    --save_dir checkpoints/config_D

In [ ]:
# ── 10. Results summary ───────────────────────────────────────────────────────
# The train script writes {run_name}_results.json per checkpoint dir.
import json, glob, os

configs = {
    'A — GW default theta':      'checkpoints/config_A',
    'B — GW language-tuned':     'checkpoints/config_B',
    'C — minGRU baseline':       'checkpoints/config_C',
    'D — GW log-spaced theta':   'checkpoints/config_D',
}

print(f"{'Config':<28} {'Best val BPC':>14} {'Test BPC':>10}")
print('-' * 54)

for label, ckpt_dir in configs.items():
    results_files = glob.glob(os.path.join(ckpt_dir, '*_results.json'))
    if not results_files:
        print(f"{label:<28} {'(not run)':>14}")
        continue
    with open(results_files[0]) as f:
        r = json.load(f)
    best_bpc = r.get('best_val_bpc', float('nan'))
    test_bpc = r.get('test_bpc', float('nan'))
    print(f"{label:<28} {best_bpc:>14.4f} {test_bpc:>10.4f}")

In [ ]:
# ── 11. Optional: copy checkpoints back to Drive ─────────────────────────────
import shutil

DRIVE_RESULTS = '/content/drive/MyDrive/continuous-language/results'
os.makedirs(DRIVE_RESULTS, exist_ok=True)
shutil.copytree('checkpoints', os.path.join(DRIVE_RESULTS, 'checkpoints'), dirs_exist_ok=True)
print('Checkpoints saved to Drive.')